## Create Attribute File ##

### Write snakemake file ###

This block of code writes the snakemake file

In [3]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

def parse_param_file(filepath):
    """Parse the parameter file and return a dict of toRun values."""
    df = pd.read_csv(filepath, sep="|")
    return df.set_index("name")["toRun"].astype(float).to_dict()

def collect_all_params(base_dir, filename, hru_gru_mapping):
    """
    Walk through base_dir, find subdirs named "hruID_{hru_id}", 
    parse existing param files, then ensure every mapped HRU_ID 
    is present (filling missing with -9999).

    Returns:
      - param_dicts: {hru_id: {param_name: value, ...}, ...}
      - param_names: sorted list of all parameter names
      - hru_ids:   sorted list of all hru_ids
    """
    param_dicts = {}
    all_params = set()

    # Phase 1: parse only existing files, build all_params
    for subdir in os.listdir(base_dir):
        full_path = os.path.join(base_dir, subdir)
        if not os.path.isdir(full_path):
            continue
        # Expect subdir name: hruID_{hru_id}
        try:
            pieces = subdir.split('_')
            if len(pieces) != 2 or pieces[0] != 'hruID':
                continue
            hru_id = int(pieces[1])
        except (IndexError, ValueError):
            # skip folders not matching pattern or invalid id
            continue

        # Find the corresponding GRU for this HRU (GRU → HRU mapping given, need HRU→GRU?)
        # Since hru_gru_mapping is gruId → hruId, let's invert it for fast lookup
        gru_for_hru = {v: k for k, v in hru_gru_mapping.items()}
        gru_id = gru_for_hru.get(hru_id, None)
        if gru_id is None:
            # Only process HRUs that are mapped from GRUs
            continue

        file_path = os.path.join(full_path, filename)
        if os.path.isfile(file_path):
            values = parse_param_file(file_path)
            param_dicts[hru_id] = values
            all_params.update(values.keys())

    # Phase 2: ensure every hru_id has an entry, filling missing with -9999
    param_names = sorted(all_params)
    hru_ids = sorted(hru_gru_mapping.values())
    for hru in hru_ids:
        if hru not in param_dicts:
            # no directory or no file for this hru → all params = -9999
            param_dicts[hru] = {p: -9999 for p in param_names}

    return param_dicts, param_names, hru_ids

def create_param_netcdf(output_path, param_dicts, param_names, hru_ids):
    """Create NetCDF file with one HRU dimension and a variable per parameter."""
    n_hru = len(hru_ids)
    ds = xr.Dataset()

    # coordinate for indexing
    ds.coords["hru"] = ("hru", np.arange(n_hru))
    # the true HRU IDs
    ds["hruId"] = ("hru", hru_ids)
    ds["hruId"].attrs["units"] = "-"
    ds["hruId"].attrs["long_name"] = "Hydrological Response Unit ID"

    # one data variable per parameter
    for param in param_names:
        vals = [param_dicts[hru].get(param, -9999) for hru in hru_ids]
        ds[param] = ("hru", vals)

    # global attributes
    ds.attrs["Author"] = "Created by SUMMA workflow scripts"
    ds.attrs["History"] = f"Created {datetime.utcnow():%Y/%m/%d %H:%M:%S}"
    ds.attrs["Purpose"] = "Trial parameter .nc file for initial SUMMA runs"

    ds.to_netcdf(output_path)
    print(f"NetCDF created at {output_path}")

def create_attribute_file(base_dir, filename, output_nc, mapping_csv):
    """Main entry: read mapping, collect params, write NetCDF."""
    hru_gru_df = pd.read_csv(mapping_csv)
    # map from gruId → hruId
    hru_gru_mapping = dict(zip(hru_gru_df['gruId'], hru_gru_df['hruId']))

    param_dicts, param_names, hru_ids = collect_all_params(
        base_dir, filename, hru_gru_mapping
    )
    create_param_netcdf(output_nc, param_dicts, param_names, hru_ids)

# Example usage:
catchment = 'tuolumne'
base_dir = f'/anvil/projects/x-ees240082/users/nicolas-vasquez/SnowStations/output/{catchment}/'
mapping_csv = f'/anvil/projects/x-ees240082/users/dcasson/gpep/{catchment}/summa/hru_gru_mapping.csv'

filename = 'RF_BestParamSet_targetMetric_TSSv1.txt'
trial_params_nc = f'/anvil/projects/x-ees240082/users/dcasson/gpep/{catchment}/summa/TSS_trialParams.nc'


create_attribute_file(base_dir, filename, trial_params_nc, mapping_csv)
        

NetCDF created at /anvil/projects/x-ees240082/users/dcasson/gpep/tuolumne/summa/TSS_trialParams.nc
